# 02 — Generate products

**Business objective:** we need a product catalog with realistic categories and
a cost/price spread, so margin-related questions ("which category drives most
revenue") are answerable later.

**What we're generating:** 200 products across retail categories, each with a
unit cost and unit price (price always above cost, with a plausible margin).

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np
from faker import Faker

rng = np.random.default_rng(cfg.SEED + 1)
fake = Faker()
Faker.seed(cfg.SEED + 1)

## Generation logic

Each category has a realistic price band (electronics expensive, groceries cheap).
Margin is randomized per product within a plausible 20-55% range.

In [2]:
categories = {
    "Electronics": (50, 1200),
    "Apparel": (10, 150),
    "Home & Kitchen": (15, 400),
    "Groceries": (2, 40),
    "Sporting Goods": (10, 300),
    "Toys": (5, 100),
    "Beauty": (5, 80),
}

cat_choices = rng.choice(list(categories.keys()), size=cfg.N_PRODUCTS)

def price_for(cat):
    lo, hi = categories[cat]
    return round(float(rng.uniform(lo, hi)), 2)

products = pd.DataFrame({
    "product_id": range(1, cfg.N_PRODUCTS + 1),
    "product_name": [fake.unique.catch_phrase() for _ in range(cfg.N_PRODUCTS)],
    "category": cat_choices,
})
products["unit_price"] = products["category"].map(price_for)
margin = rng.uniform(0.20, 0.55, size=cfg.N_PRODUCTS)
products["unit_cost"] = (products["unit_price"] * (1 - margin)).round(2)
products.head()

,product_id,product_name,category,unit_price,unit_cost
0,1,Automated executive standardization,Groceries,15.79,12.16
1,2,Virtual client-server Local Area Network,Sporting Goods,280.62,199.69
2,3,Multi-layered static standardization,Home & Kitchen,388.25,176.83
3,4,Cross-platform logistical policy,Electronics,355.24,185.20
4,5,Proactive real-time adapter,Sporting Goods,221.90,106.95


## Sample output & distribution check

In [3]:
print(products["category"].value_counts())
print()
print(products[["unit_cost", "unit_price"]].describe())

category
Sporting Goods    35
Groceries         30
Home & Kitchen    30
Beauty            29
Apparel           26
Electronics       25
Toys              25
Name: count, dtype: int64

        unit_cost   unit_price
count  200.000000   200.000000
mean    97.468900   158.486400
std    132.424229   220.588601
min      3.050000     5.100000
25%     19.015000    33.532500
50%     45.550000    75.955000
75%    128.577500   188.862500
max    719.230000  1168.430000


## Validation

In [4]:
assert products["product_id"].is_unique
assert (products["unit_price"] > products["unit_cost"]).all(), "price must exceed cost"
assert products.isnull().sum().sum() == 0
print("All validation checks passed.")

All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "products.csv"
products.to_csv(out_path, index=False)
print(f"Wrote {len(products):,} rows to {out_path}")

Wrote 200 rows to /home/claude/project/eaida/data/raw/products.csv


## Summary

Generated 200 products across 7 categories with realistic category-appropriate
pricing and a 20-55% margin band. Price always exceeds cost. Saved to
`data/raw/products.csv` for use by `order_items` and `inventory`.